## Azure Data Engineering Take-Home Assignment
# Project Overview
This project builds an end-to-end data pipeline that extracts data from API endpoints, processes the data using PySpark, and generates business insights such as customer revenue and revenue by state.

This project simulates a real-world data engineering workflow including:

Data ingestion from API \
Data transformation using PySpark \
Data aggregation \
Data export for analytics

The pipeline follows these steps:

1. Extract data from API endpoints
2. Handle pagination to retrieve all records
3. Load data into PySpark DataFrames
4. Transform and join datasets
5. Calculate business metrics
6. Export final dataset

In a production environment, this pipeline could be implemented using:
- Azure Data Factory (orchestration)
- Azure Data Lake (storage)
- Azure Synapse (analytics)

In [1]:
import requests
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum
import shutil

This function fetches data from API endpoints and handles pagination to retrieve all records.

In [2]:
import time

# Create Spark session
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("blue-owls-assessment") \
    .getOrCreate()

print("Spark Ready:", spark.version)

Spark Ready: 3.5.0


In [3]:
API_BASE_URL = "http://mock-api:8000"

res = requests.get(f"{API_BASE_URL}/api/v1/health")

print("Status Code:", res.status_code)
print("Response:", res.json())

Status Code: 200
Response: {'status': 'healthy', 'timestamp': '2026-03-26T07:58:25.903855+00:00'}


Authentication (Token)

In [4]:
def get_token():
    res = requests.post(
        f"{API_BASE_URL}/api/v1/auth/token",
        json={
            "username": "candidate",
            "password": "blue-owls-2026"
        }
    )
    
    if res.status_code != 200:
        raise Exception("Auth failed:", res.text)
    
    return res.json()["access_token"]

token = get_token()
print("Token received:", token[:10], "…")

Token received: eyJhbGciOi …


API Fetch Function (Pagination + Retry)

In [5]:
def fetch_all_data(endpoint, retries=3):
    all_data = []
    page = 1
    token = get_token()

    while True:
        attempt = 0

        while attempt < retries:
            headers = {"Authorization": f"Bearer {token}"}
            url = f"{API_BASE_URL}/api/v1/data/{endpoint}?page={page}"

            res = requests.get(url, headers=headers)

            # HANDLE TOKEN EXPIRY
            if res.status_code == 401:
                print(" Token expired. Refreshing...")
                token = get_token()
                continue

            if res.status_code == 200:
                break

            print(f"Retry {attempt+1} for {endpoint} page {page}")
            attempt += 1
            time.sleep(1)

        if res.status_code != 200:
            print(f" Failed: {endpoint}")
            break

        json_data = res.json()
        records = json_data["data"]

        all_data.extend(records)

        print(f"{endpoint} → Page {page} fetched ({len(records)} records)")

        if page >= json_data["pagination"]["total_pages"]:
            break

        page += 1

    print(f" Total {endpoint} records:", len(all_data))
    return all_data

In [6]:
orders = fetch_all_data("orders")
customers = fetch_all_data("customers")
products = fetch_all_data("products")
payments = fetch_all_data("payments")
order_items = fetch_all_data("order_items")

orders → Page 1 fetched (1000 records)
orders → Page 2 fetched (1000 records)
orders → Page 3 fetched (1000 records)
orders → Page 4 fetched (1000 records)
Retry 1 for orders page 5
orders → Page 5 fetched (1000 records)
Retry 1 for orders page 6
Retry 2 for orders page 6
Retry 3 for orders page 6
 Failed: orders
 Total orders records: 5000
Retry 1 for customers page 1
customers → Page 1 fetched (1000 records)
customers → Page 2 fetched (1000 records)
customers → Page 3 fetched (1000 records)
customers → Page 4 fetched (1000 records)
customers → Page 5 fetched (1000 records)
customers → Page 6 fetched (1000 records)
customers → Page 7 fetched (1000 records)
customers → Page 8 fetched (1000 records)
customers → Page 9 fetched (1000 records)
Retry 1 for customers page 10
Retry 2 for customers page 10
customers → Page 10 fetched (1000 records)
customers → Page 11 fetched (1000 records)
customers → Page 12 fetched (1000 records)
customers → Page 13 fetched (1000 records)
customers → Page 1

Fetch All Tables (Bronze Layer Raw Data)

In [7]:
df_orders = spark.createDataFrame(orders)
df_customers = spark.createDataFrame(customers)
df_products = spark.createDataFrame(products)
df_payments = spark.createDataFrame(payments)
df_order_items = spark.createDataFrame(order_items)

df_orders.show(5)

+--------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+------------------------+------------+
|         customer_id|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|            order_id|order_purchase_timestamp|order_status|
+--------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+------------------------+------------+
|9ef432eb625129730...|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|e481f51cbdc54678b...|     2017-10-02 10:56:33|   delivered|
|b0830fb4747a6c6d2...|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|53cdb2fc8bc7dce0b...|     2018-07-24 20:41:37|   delivered|
|41ce2a54c0b03bf34...|2018-08-08 08:55:23|   

Data Cleaning (Silver Layer)

In [8]:
df_payments = df_payments.withColumn(
    "payment_value",
    col("payment_value").cast("double")
)

df_orders = df_orders.withColumn(
    "order_purchase_timestamp",
    col("order_purchase_timestamp").cast("timestamp")
)

Save Bronze Data

In [9]:
df_orders.write.mode("overwrite").parquet("submission/output/bronze/orders")
df_customers.write.mode("overwrite").parquet("submission/output/bronze/customers")
df_products.write.mode("overwrite").parquet("submission/output/bronze/products")
df_payments.write.mode("overwrite").parquet("submission/output/bronze/payments")
df_order_items.write.mode("overwrite").parquet("submission/output/bronze/order_items")

In [10]:
df_order_items.printSchema()

root
 |-- freight_value: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- price: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)



In [11]:
df_dim_sellers = df_order_items.select("seller_id").dropDuplicates()

Build Silver Layer

In [12]:
df_orders_clean = df_orders.dropDuplicates()
df_customers_clean = df_customers.dropDuplicates()
df_products_clean = df_products.dropDuplicates()
df_payments_clean = df_payments.dropDuplicates()
df_order_items_clean = df_order_items.dropDuplicates()

In [13]:
df_orders_clean.write.mode("overwrite").csv("submission/output/silver/orders_clean")
df_customers_clean.write.mode("overwrite").csv("submission/output/silver/customers_clean")
df_products_clean.write.mode("overwrite").csv("submission/output/silver/products_clean")
df_payments_clean.write.mode("overwrite").csv("submission/output/silver/payments_clean")
df_order_items_clean.write.mode("overwrite").csv("submission/output/silver/order_items_clean")

Build Gold Layer (Star Schema)

In [14]:
fact_order_items = df_orders_clean \
    .join(df_order_items_clean, "order_id") \
    .join(df_payments_clean, "order_id")

Dimension Tables

In [15]:
df_dim_customers = df_customers_clean
df_dim_products = df_products_clean
df_dim_sellers = df_order_items.select("seller_id").dropDuplicates()

Save Gold Layer

In [16]:
fact_order_items.write.mode("overwrite").parquet("submission/output/gold/fact_order_items ")
df_dim_customers.write.mode("overwrite").parquet("submission/output/gold/dim_customers")
df_dim_products.write.mode("overwrite").parquet("submission/output/gold/dim_products")
df_dim_sellers.write.mode("overwrite").parquet("submission/output/gold/dim_sellers")


Revenue by Customer

In [17]:
df_revenue = fact_order_items.groupBy("customer_id").agg(
    sum("payment_value").alias("total_spent")
)

df_revenue.orderBy("total_spent", ascending=False).show(10)

+--------------------+-----------+
|         customer_id|total_spent|
+--------------------+-----------+
|9bba9922e5c4a9433...|    5080.32|
|4b24f5ef0c134fe4d...|    2772.36|
|cc89bb0f40fb83134...|     905.03|
|3e13d7562141418f0...|     785.24|
|4b83fd2609057103e...|     708.95|
|a820146409b46b535...|      705.0|
|2698695709c21e561...|     607.52|
|ab5a1016ed934dc4b...|     596.27|
|a791256150a697532...|      512.9|
|4c9b48364e100b48b...|     467.32|
+--------------------+-----------+
only showing top 10 rows



Gold Layer Business Analysis – Revenue by Product

In [18]:
fact_order_items.groupBy("product_id").agg(
    sum("payment_value").alias("total_revenue")
).orderBy("total_revenue", ascending=False).show(10)

+--------------------+-----------------+
|          product_id|    total_revenue|
+--------------------+-----------------+
|58efb9b638561ce13...|5080.320000000001|
|8bb27b1d96be90b36...|          2772.36|
|f27aff266ad97e75d...|           905.03|
|a630323d497e2acc8...|           785.24|
|5f504b3a1c75b73d6...|           708.95|
|ea260f972b87224bc...|            705.0|
|1dec4c88c685d5a07...|           596.27|
|21fd3b391a97c2fed...|            512.9|
|63085bb4366ded27b...|           446.51|
|1f112631857414c63...|            436.5|
+--------------------+-----------------+
only showing top 10 rows



In [21]:
docker_compose = """
version: '3'
services:
  notebook:
    image: jupyter/pyspark-notebook
    container_name: pyspark_notebook
    ports:
      - "8888:8888"
    volumes:
      - .:/home/jovyan/submission
    environment:
      - JUPYTER_ENABLE_LAB=yes
"""

with open("submission/docker-compose.yml", "w") as f:
    f.write(docker_compose)

print("docker-compose.yml created")


docker-compose.yml created


In [22]:
readme = """
# Azure Data Engineer Pipeline Assignment

## Project Overview
This project implements a data pipeline using PySpark inside Docker. The pipeline ingests API data, processes it into Bronze, Silver, and Gold layers, and generates analytics tables.

## Project Structure
submission/
│
├── azure_data_engineer_pipeline_api_blue_owls.ipynb
├── docker-compose.yml
├── README.md
│
├── output/
│   ├── bronze/
│   ├── silver/
│   └── gold/
│
└── sql/
    ├── query_1.sql
    └── query_2.sql

## How to Run
1. Start Docker:
   docker-compose up

2. Open Jupyter Lab:
   http://localhost:8888

3. Open and run notebook:
   azure_data_engineer_pipeline_api_blue_owls.ipynb

4. Output will be generated in:
   output/bronze
   output/silver
   output/gold

## Gold Layer Tables
- fact_order_items
- dim_customers
- dim_products
- dim_sellers

## SQL Queries
SQL queries are available in submission/sql/ for analytics:
- Revenue Trend Analysis
- Seller Performance Scorecard
"""

with open("submission/README.md", "w") as f:
    f.write(readme)

print("README.md created")


README.md created
